In [11]:
# tous les bib utilise|
from transformers import pipeline , AutoTokenizer, AutoModelForSequenceClassification
from langdetect import detect, DetectorFactory
import re 
import pandas
import torch 

detection de langauge 

In [1]:
# installer le modele localement

from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "papluca/xlm-roberta-base-language-detection"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer.save_pretrained("./lang_detector")
model.save_pretrained("./lang_detector")

print("✅ Modèle sauvegardé dans ./lang_detector")

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Charger depuis le dossier local
model_path = "./local_models/lang_detector"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

detector = pipeline("text-classification", model=model, tokenizer=tokenizer)

In [ ]:
# --- Tests ---
texts = [
    # Français
    "Bonjour comment ça va",
    "c'est quoi ça",
    "je vais bien merci",
    
    # Arabe MSA
    "كيف حالك يا صديقي",
    "مرحبا كيف أنت اليوم",
    
    # Darija arabe
    "واش نتا بخير؟",
    "باهي نروحو لدار",
    "راهو مجاش بعد",
    
    # Darija latinisée
    "rabi ystur 3lina",
    "ana mchi l dar",
    "wach nta labas",
]

print(f"{'Texte':<35} {'Label':<10} {'Confiance'}")
print("-" * 60)

for t in texts:
    result = detector(t, top_k=1)[0]
    label = result["label"]
    conf = round(result["score"] * 100, 2)
    print(f"{t:<35} {label:<10} {conf}%")

In [ ]:
df = pandas.read_csv('DATA\MergedComplaints.csv')
df["language_detecter"] = df['Commentaire client'].apply(detect_language_fr_ar_derja)

In [ ]:
df.drop(columns=['Defined Class','Classe associé'] , inplace=True)
df.to_csv("DATA/df_tempo.csv")

In [7]:
df = pandas.read_csv('DATA\df_tempo.csv')
df[ (df["language_detecter"] == 'Derja') & (df["Predicted class"] == "Problème / Réclamation") ].to_csv("DATA/voirDarja.csv")
df = pandas.read_csv('DATA/voirDarja.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_6596\1887294772.py:1: SyntaxWarning: invalid escape sequence '\d'
  df = pandas.read_csv('DATA\df_tempo.csv')


In [ ]:
#df_filtre = df[df['Commentaire client'].str.contains(r'\b3ndna\b', regex=True, na=False)]

#df_filtre = df[df['Commentaire client'].str.contains(r'(?i)(?<!\w)ça fait(?!\w)', regex=True, na=False)]

#df_filtre = df[df['Commentaire client'].str.contains(r'(^|\s)من(\s|$)', regex=True, na=False)]

# On met directement les plages de chiffres arabes et persans
filtre_chiffres = df[df['Commentaire client'].str.contains(r'[٠-٩۰-۹]', regex=True, na=False)]


# Supprime la limite d'affichage des lignes
pandas.set_option('display.max_rows', None)

# Affiche votre dataframe filtré
print(filtre_chiffres)

In [ ]:
df["Commentaire client"].loc[]

detecter le temp d'attente 

In [4]:

# comment savoir que la date extract est une date d'attente : extcract only from reclamation category 
# apres avoir detecter la langue si fr / ar --->


# example en arabe (same for french) cole et non cole
# 2 ans ,منذ شهر ,  عامين ,  5 ايام,  4 اشهر ,البارح , يومين , شهرين , عام , من البارح , 16 يوم , المساء , اسبوع , الايام , باليوم 9 , J + 24 ,  48 ساعة , ça fait 2 jours , اكثر من شهر
# example en darja 
# 3ndi 2 mois ,  3ndna 3ans w 7 chhr , rah 2mois,  7 chhor , 3am , سمانة , man lkhmis , مالبارح , من بارح  , بارحة,m lberh ,lberh, السمانة , w 2 ans , عندو 6اشهر 

# IMPORTANT POINT :
# pour check tout ca avec les patterens on doit negligier espace avant et apres
# il existe des chiffres en arabe 
# si le langauge detcter est darja alors utiliser les pattern arabe et fr et darja 


# convertire chiffre arabe en fr 
def arNumbers_toFr(text:str):
    arabic_nums = "٠١٢٣٤٥٦٧٨٩"
    western_nums = "0123456789"
    trans = str.maketrans(arabic_nums, western_nums)
    return text.translate(trans)

# extract a number and word from a specifique pattern
def extract_all_from_pat(text, patterns):
    for p in patterns:
        match = re.search(p, text, re.IGNORECASE)
        if match:
            groups = match.groups()
            # Cas standard : (\d+)\s*(unité) -> deux groupes
            if len(groups) >= 2:
                try:
                    val = int(groups[0])
                    unit = str(groups[1])
                    return val, unit
                except (ValueError, IndexError):
                    continue
            # CAS SPECIAL ,  j+24 : un seul groupe (\d+)
            elif len(groups) == 1:
                return int(groups[0]), "j+"
    return None, None


def detect_waiting_time(text_cleaned) -> (int , str) : 
    
    text_cleaned = arNumbers_toFr(text_cleaned)
    text_cleaned = text_cleaned.lower()

    lang = detect_language_fr_ar_derja(text_cleaned)

    # definir les patterns 
    patterns_fr = [
    r"(\d+)\s*(?:jour|jours)",
    r"(\d+)\s*(?:mois)",
    r"(\d+)\s*(?:an|ans)",
    r"(\d+)\s*(?:heure|heures)",
    r"(\d+)\s*(?:semaine|semaines)",
    r"j\+?\s*(\d+)" ]

    patterns_ar = [
    r"\d+\s*(يوم|ايام)",
    r"\d+\s*(شهر|اشهر)",
    r"\d+\s*(عام|اعوام|سنوات)",
    r"\d+\s*(ساعة|ساعات)",
    r"\d+\s*(اسبوع|اسابيع)"]
    
    pattern_darja = [
    r"(3ndi|3ndna|3ndha|rah|men|man|ml|m|w|3and|"
    r"\d*\s*(chhr|chhor|snin|3am|nhar|nhr|سمانة|smana|smanat|swayr|sara|سوايع)|"
    r"lberh|m\s*lberh|barh|malbarh|lebereh|lbrh)"]
    
    # --- LOGIQUE DE DÉTECTION ---
    val, unit = None, None
    if lang == "fr":
        val, unit = extract_fromPat(text_cleaned, patterns_fr)
    elif lang == "ar":
        val, unit = extract_fromPat(text_cleaned, patterns_ar)
    elif lang == "darja":
        # Test successif
        for pattern in [patterns_fr, patterns_ar, patterns_darja]:
            val, unit = extract_fromPat(text_cleaned, pattern)
            if val is not None:
                break

    return val, unit

EXTRCAT ENTITIES